In [27]:
import mysql.connector
import pandas as pd
import hashlib
import warnings
warnings.filterwarnings('ignore')

In [28]:
print("\n" + "="*80)
print(f"📊 Database Configuration")
print("="*80)



📊 Database Configuration


In [29]:
DB_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'Shalu0.34',  
    'database': 'sales_intelligence_hub'
}

def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

def run_query(query, title):
    conn = get_connection()
    print("\n" + "="*80)
    print(f"📊 {title}")
    print("="*80)
    try:
        result = pd.read_sql(query, conn)
        print(result.to_string(index=False))
        print(f"\n✅ Rows: {len(result)}")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        conn.close()
    return result

print("✅ Setup complete!")

✅ Setup complete!


In [23]:
print("="*80)
print("📂 LOADING CSV DATA INTO MYSQL TABLES")
print("="*80)

📂 LOADING CSV DATA INTO MYSQL TABLES


In [ ]:

conn = get_connection()
cursor = conn.cursor()

print("="*80)
print("📂 LOADING CSV DATA INTO MYSQL TABLES")
print("="*80)

# 1. Load branches
print("\n📁 Loading branches.csv...")
branches_data = [
    (1, 'Chennai', 'Arun Kumar'),
    (2, 'Bangalore', 'Ravi Shankar'),
    (3, 'Hyderabad', 'Suresh Reddy'),
    (4, 'Delhi', 'Neha Sharma'),
    (5, 'Mumbai', 'Rahul Mehta'),
    (6, 'Pune', 'Amit Patil'),
    (7, 'Kolkata', 'Subham Ghosh'),
    (8, 'Ahmedabad', 'Raj Patel')
]
cursor.execute("DELETE FROM branches")
for row in branches_data:
    cursor.execute("INSERT INTO branches (branch_id, branch_name, branch_admin_name) VALUES (%s, %s, %s)", row)
print(f"   ✅ Loaded {len(branches_data)} branches")

# 2. Load customer_sales
print("\n📁 Loading customer_sales.csv...")
df_sales = pd.read_csv('customer_sales.csv')
cursor.execute("DELETE FROM customer_sales")
for _, row in df_sales.iterrows():
    cursor.execute("""
        INSERT INTO customer_sales (sale_id, branch_id, date, name, mobile_number, product_name, gross_sales, received_amount, status)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, (row['sale_id'], row['branch_id'], row['date'], row['name'], 
          row['mobile_number'], row['product_name'], row['gross_sales'], 
          row['received_amount'], row['status']))
print(f"   ✅ Loaded {len(df_sales)} sales records")

# 3. Load payment_splits
print("\n📁 Loading payment_splits.csv...")
df_payments = pd.read_csv('payment_splits.csv')
cursor.execute("DELETE FROM payment_splits")
for _, row in df_payments.iterrows():
    cursor.execute("""
        INSERT INTO payment_splits (payment_id, sale_id, payment_date, amount_paid, payment_method)
        VALUES (%s, %s, %s, %s, %s)
    """, (row['payment_id'], row['sale_id'], row['payment_date'], row['amount_paid'], row['payment_method']))
print(f"   ✅ Loaded {len(df_payments)} payment records")

# 4. Load users
print("\n📁 Loading users.csv...")
df_users = pd.read_csv('users.csv')
cursor.execute("DELETE FROM users")
for _, row in df_users.iterrows():
    hashed_pwd = hashlib.sha256(row['password'].encode()).hexdigest()
    branch_id = row['branch_id'] if pd.notna(row['branch_id']) else None
    cursor.execute("""
        INSERT INTO users (user_id, username, password, branch_id, role, email)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (row['user_id'], row['username'], hashed_pwd, branch_id, row['role'], row['email']))
print(f"   ✅ Loaded {len(df_users)} users")

conn.commit()
cursor.close()
conn.close()

print("\n" + "="*80)
print("✅ ALL CSV DATA LOADED SUCCESSFULLY!")
print("="*80)

📂 LOADING CSV DATA INTO MYSQL TABLES

📁 Loading branches.csv...
   ✅ Loaded 8 branches

📁 Loading customer_sales.csv...
   ✅ Loaded 1000 sales records

📁 Loading payment_splits.csv...
   ✅ Loaded 2011 payment records

📁 Loading users.csv...
   ✅ Loaded 9 users

✅ ALL CSV DATA LOADED SUCCESSFULLY!


In [26]:
print("="*80)
print("📊 VERIFYING DATA IN TABLES")
print("="*80)

📊 VERIFYING DATA IN TABLES


In [13]:
conn = get_connection()
cursor = conn.cursor()

print("\n📊 VERIFYING DATA IN TABLES:")
print("-"*40)

cursor.execute("SELECT COUNT(*) FROM branches")
print(f"   branches: {cursor.fetchone()[0]} records")

cursor.execute("SELECT COUNT(*) FROM customer_sales")
print(f"   customer_sales: {cursor.fetchone()[0]} records")

cursor.execute("SELECT COUNT(*) FROM payment_splits")
print(f"   customer_sales: {cursor.fetchone()[0]} records")

cursor.execute("SELECT COUNT(*) FROM users")
print(f"   users: {cursor.fetchone()[0]} records")

cursor.close()
conn.close()


📊 VERIFYING DATA IN TABLES:
----------------------------------------
   branches: 8 records
   customer_sales: 1000 records
   customer_sales: 2011 records
   users: 9 records


In [18]:
# Step 3: Run all SQL analysis queries
print("\n" + "="*80)
print("📊 RUNNING SQL ANALYSIS QUERIES")
print("="*80)


📊 RUNNING SQL ANALYSIS QUERIES


In [19]:
# BASIC QUERIES 
# Query 1: Retrieve all records from customer_sales table
query1 = "SELECT * FROM customer_sales;"
run_query(query1, "Query 1: All Customer Sales Records")


📊 Query 1: All Customer Sales Records
 sale_id  branch_id       date          name mobile_number product_name  gross_sales  received_amount  pending_amount status
       1          2 2024-01-02    Customer_1    9800000001           DS      40000.0           7296.0         32704.0   Open
       2          5 2024-01-03    Customer_2    9800000002           BA      30000.0           7314.0         22686.0   Open
       3          4 2024-01-04    Customer_3    9800000003           DA      35000.0           5697.0         29303.0   Open
       4          2 2024-01-05    Customer_4    9800000004          FSD      45000.0           1739.0         43261.0   Open
       5          7 2024-01-06    Customer_5    9800000005           DS      40000.0          18231.0         21769.0   Open
       6          1 2024-01-07    Customer_6    9800000006          FSD      45000.0          27696.0         17304.0   Open
       7          4 2024-01-08    Customer_7    9800000007           BA      30000.0  

,sale_id,branch_id,date,name,mobile_number,product_name,gross_sales,received_amount,pending_amount,status
0,1,2,2024-01-02,Customer_1,9800000001,DS,40000.0,7296.0,32704.0,Open
1,2,5,2024-01-03,Customer_2,9800000002,BA,30000.0,7314.0,22686.0,Open
2,3,4,2024-01-04,Customer_3,9800000003,DA,35000.0,5697.0,29303.0,Open
3,4,2,2024-01-05,Customer_4,9800000004,FSD,45000.0,1739.0,43261.0,Open
4,5,7,2024-01-06,Customer_5,9800000005,DS,40000.0,18231.0,21769.0,Open
...,...,...,...,...,...,...,...,...,...,...
995,996,8,2024-09-23,Customer_996,9800000996,DS,40000.0,37482.0,2518.0,Open
996,997,6,2024-09-24,Customer_997,9800000997,SQL,25000.0,1389.0,23611.0,Open
997,998,6,2024-09-25,Customer_998,9800000998,SQL,25000.0,12963.0,12037.0,Open
998,999,7,2024-09-26,Customer_999,9800000999,DA,35000.0,28806.0,6194.0,Open


In [54]:
# Query 2: Retrieve all records from branches table
query2 = "SELECT * FROM branches;"
run_query(query2, "Query 2: All Branches")


📊 Query 2: All Branches
 branch_id branch_name branch_admin_name
         1     Chennai        Arun Kumar
         2   Bangalore      Ravi Shankar
         3   Hyderabad      Suresh Reddy
         4       Delhi       Neha Sharma
         5      Mumbai       Rahul Mehta
         6        Pune        Amit Patil
         7     Kolkata      Subham Ghosh
         8   Ahmedabad         Raj Patel

✅ Rows returned: 8


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [55]:
# Query 3: Retrieve all records from payment_splits table
query3 = "SELECT * FROM payment_splits;"
run_query(query3, "Query 3: All Payment Splits")


📊 Query 3: All Payment Splits
 payment_id  sale_id payment_date  amount_paid payment_method
          1        1   2024-01-06       7296.0           Cash
          2        2   2024-01-04       7314.0           Card
          3        3   2024-01-04       3457.0           Cash
          4        3   2024-01-07        384.0           Cash
          5        3   2024-01-12       1856.0           Card
          6        4   2024-01-15        408.0           Card
          7        4   2024-01-11       1117.0           Cash
          8        4   2024-01-12        214.0           Card
          9        5   2024-01-08      18231.0           Card
         10        6   2024-01-09       9106.0           Cash
         11        6   2024-01-12      18590.0           Cash
         12        7   2024-01-13        397.0            UPI
         13        7   2024-01-17       2642.0            UPI
         14        8   2024-01-16      26447.0           Card
         15        9   2024-01-18      

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [56]:
# Query 4: Display all sales with status = 'Open'
query4 = "SELECT * FROM customer_sales WHERE status = 'Open';"
run_query(query4, "Query 4: Open Sales (Pending Payments)")


📊 Query 4: Open Sales (Pending Payments)
 sale_id  branch_id       date          name mobile_number product_name  gross_sales  received_amount  pending_amount status
       1          2 2024-01-02    Customer_1    9800000001           DS      40000.0           7296.0         32704.0   Open
       2          5 2024-01-03    Customer_2    9800000002           BA      30000.0           7314.0         22686.0   Open
       3          4 2024-01-04    Customer_3    9800000003           DA      35000.0           5697.0         29303.0   Open
       4          2 2024-01-05    Customer_4    9800000004          FSD      45000.0           1739.0         43261.0   Open
       5          7 2024-01-06    Customer_5    9800000005           DS      40000.0          18231.0         21769.0   Open
       6          1 2024-01-07    Customer_6    9800000006          FSD      45000.0          27696.0         17304.0   Open
       7          4 2024-01-08    Customer_7    9800000007           BA      30000.

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [57]:
# Query 5: Retrieve all sales belonging to Chennai branch
query5 = """SELECT cs.*, b.branch_name 
            FROM customer_sales cs
            JOIN branches b ON cs.branch_id = b.branch_id
            WHERE b.branch_name = 'Chennai';"""
run_query(query5, "Query 5: Chennai Branch Sales")


📊 Query 5: Chennai Branch Sales
 sale_id  branch_id       date         name mobile_number product_name  gross_sales  received_amount  pending_amount status branch_name
       6          1 2024-01-07   Customer_6    9800000006          FSD      45000.0          27696.0         17304.0   Open     Chennai
       8          1 2024-01-09   Customer_8    9800000008           BA      30000.0          26447.0          3553.0   Open     Chennai
      11          1 2024-01-12  Customer_11    9800000011           DA      35000.0          24911.0         10089.0   Open     Chennai
      23          1 2024-01-24  Customer_23    9800000023           BA      30000.0          23616.0          6384.0   Open     Chennai
      35          1 2024-02-05  Customer_35    9800000035           BA      30000.0           7216.0         22784.0   Open     Chennai
      36          1 2024-02-06  Customer_36    9800000036           AI      48000.0          41359.0          6641.0   Open     Chennai
      57       

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [58]:
# AGGREGATION QUERIES
# Query 6: Total gross sales across all branches
query6 = "SELECT SUM(gross_sales) AS total_gross_sales FROM customer_sales;"
run_query(query6, "Query 6: Total Gross Sales (All Branches)")


📊 Query 6: Total Gross Sales (All Branches)
 total_gross_sales
        36768000.0

✅ Rows returned: 1


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [59]:
# Query 7: Total received amount across all sales
query7 = "SELECT SUM(received_amount) AS total_received_amount FROM customer_sales;"
run_query(query7, "Query 7: Total Received Amount")


📊 Query 7: Total Received Amount
 total_received_amount
            19028113.0

✅ Rows returned: 1


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [60]:
# Query 8: Total pending amount across all sales
query8 = "SELECT SUM(pending_amount) AS total_pending_amount FROM customer_sales;"
run_query(query8, "Query 8: Total Pending Amount")


📊 Query 8: Total Pending Amount
 total_pending_amount
           17739887.0

✅ Rows returned: 1


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [61]:
# Query 9: Count total number of sales per branch
query9 = """SELECT b.branch_name, COUNT(cs.sale_id) AS total_sales
            FROM branches b
            LEFT JOIN customer_sales cs ON b.branch_id = cs.branch_id
            GROUP BY b.branch_name;"""
run_query(query9, "Query 9: Sales Count Per Branch")


📊 Query 9: Sales Count Per Branch
branch_name  total_sales
    Chennai          110
  Bangalore          133
  Hyderabad          106
      Delhi          134
     Mumbai          136
       Pune          120
    Kolkata          133
  Ahmedabad          128

✅ Rows returned: 8


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [62]:
# Query 10: Average gross sales amount
query10 = "SELECT AVG(gross_sales) AS avg_gross_sales FROM customer_sales;"
run_query(query10, "Query 10: Average Gross Sales Amount")


📊 Query 10: Average Gross Sales Amount
 avg_gross_sales
         36768.0

✅ Rows returned: 1


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [63]:
# JOIN-BASED QUERIES
# Query 11: Sales details with branch name
query11 = """SELECT cs.sale_id, b.branch_name, cs.date, cs.name, 
                    cs.product_name, cs.gross_sales, cs.received_amount, 
                    cs.pending_amount, cs.status
            FROM customer_sales cs
            JOIN branches b ON cs.branch_id = b.branch_id;"""
run_query(query11, "Query 11: Sales Details with Branch Name")


📊 Query 11: Sales Details with Branch Name
 sale_id branch_name       date          name product_name  gross_sales  received_amount  pending_amount status
       6     Chennai 2024-01-07    Customer_6          FSD      45000.0          27696.0         17304.0   Open
       8     Chennai 2024-01-09    Customer_8           BA      30000.0          26447.0          3553.0   Open
      11     Chennai 2024-01-12   Customer_11           DA      35000.0          24911.0         10089.0   Open
      23     Chennai 2024-01-24   Customer_23           BA      30000.0          23616.0          6384.0   Open
      35     Chennai 2024-02-05   Customer_35           BA      30000.0           7216.0         22784.0   Open
      36     Chennai 2024-02-06   Customer_36           AI      48000.0          41359.0          6641.0   Open
      57     Chennai 2024-02-27   Customer_57           ML      42000.0          36896.0          5104.0   Open
      63     Chennai 2024-03-04   Customer_63          FSD  

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [64]:
# Query 12: Sales details with total payment received from payment_splits
query12 = """SELECT cs.sale_id, cs.name, cs.gross_sales, 
                    COALESCE(SUM(ps.amount_paid), 0) AS total_paid_via_splits
            FROM customer_sales cs
            LEFT JOIN payment_splits ps ON cs.sale_id = ps.sale_id
            GROUP BY cs.sale_id, cs.name, cs.gross_sales;"""
run_query(query12, "Query 12: Sales with Payment Split Totals")



📊 Query 12: Sales with Payment Split Totals
 sale_id          name  gross_sales  total_paid_via_splits
       1    Customer_1      40000.0                 7296.0
       2    Customer_2      30000.0                 7314.0
       3    Customer_3      35000.0                 5697.0
       4    Customer_4      45000.0                 1739.0
       5    Customer_5      40000.0                18231.0
       6    Customer_6      45000.0                27696.0
       7    Customer_7      30000.0                 3039.0
       8    Customer_8      30000.0                26447.0
       9    Customer_9      30000.0                 4090.0
      10   Customer_10      42000.0                23700.0
      11   Customer_11      35000.0                24911.0
      12   Customer_12      48000.0                23283.0
      13   Customer_13      35000.0                 4679.0
      14   Customer_14      48000.0                41943.0
      15   Customer_15      45000.0                26290.0
      16   

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [65]:
# Query 13: Branch-wise total gross sales
query13 = """SELECT b.branch_name, SUM(cs.gross_sales) AS total_gross_sales
            FROM branches b
            JOIN customer_sales cs ON b.branch_id = cs.branch_id
            GROUP BY b.branch_name
            ORDER BY total_gross_sales DESC;"""
run_query(query13, "Query 13: Branch-wise Total Gross Sales")


📊 Query 13: Branch-wise Total Gross Sales
branch_name  total_gross_sales
    Kolkata          5101000.0
      Delhi          4914000.0
     Mumbai          4853000.0
  Bangalore          4710000.0
  Ahmedabad          4695000.0
       Pune          4558000.0
    Chennai          4128000.0
  Hyderabad          3809000.0

✅ Rows returned: 8


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [66]:
# Query 14: Sales with payment method used
query14 = """SELECT cs.sale_id, cs.name, ps.payment_method, ps.amount_paid
            FROM customer_sales cs
            JOIN payment_splits ps ON cs.sale_id = ps.sale_id;"""
run_query(query14, "Query 14: Sales with Payment Method")


📊 Query 14: Sales with Payment Method
 sale_id          name payment_method  amount_paid
       1    Customer_1           Cash       7296.0
       2    Customer_2           Card       7314.0
       3    Customer_3           Cash       3457.0
       3    Customer_3           Cash        384.0
       3    Customer_3           Card       1856.0
       4    Customer_4           Card        408.0
       4    Customer_4           Cash       1117.0
       4    Customer_4           Card        214.0
       5    Customer_5           Card      18231.0
       6    Customer_6           Cash       9106.0
       6    Customer_6           Cash      18590.0
       7    Customer_7            UPI        397.0
       7    Customer_7            UPI       2642.0
       8    Customer_8           Card      26447.0
       9    Customer_9            UPI        323.0
       9    Customer_9           Card       3767.0
      10   Customer_10           Cash       6301.0
      10   Customer_10           Cash      

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [67]:
# Query 15: Sales with branch admin name
query15 = """SELECT cs.sale_id, cs.name, b.branch_name, b.branch_admin_name
            FROM customer_sales cs
            JOIN branches b ON cs.branch_id = b.branch_id;"""
run_query(query15, "Query 15: Sales with Branch Admin Name")


📊 Query 15: Sales with Branch Admin Name
 sale_id          name branch_name branch_admin_name
       6    Customer_6     Chennai        Arun Kumar
       8    Customer_8     Chennai        Arun Kumar
      11   Customer_11     Chennai        Arun Kumar
      23   Customer_23     Chennai        Arun Kumar
      35   Customer_35     Chennai        Arun Kumar
      36   Customer_36     Chennai        Arun Kumar
      57   Customer_57     Chennai        Arun Kumar
      63   Customer_63     Chennai        Arun Kumar
      79   Customer_79     Chennai        Arun Kumar
      80   Customer_80     Chennai        Arun Kumar
     101  Customer_101     Chennai        Arun Kumar
     106  Customer_106     Chennai        Arun Kumar
     113  Customer_113     Chennai        Arun Kumar
     115  Customer_115     Chennai        Arun Kumar
     121  Customer_121     Chennai        Arun Kumar
     136  Customer_136     Chennai        Arun Kumar
     144  Customer_144     Chennai        Arun Kumar
    

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [68]:
# FINANCIAL TRACKING QUERIES
# Query 16: Sales where pending amount > 5000
query16 = """SELECT cs.*, b.branch_name
            FROM customer_sales cs
            JOIN branches b ON cs.branch_id = b.branch_id
            WHERE cs.pending_amount > 5000;"""
run_query(query16, "Query 16: Sales with Pending Amount > 5000")



📊 Query 16: Sales with Pending Amount > 5000
 sale_id  branch_id       date          name mobile_number product_name  gross_sales  received_amount  pending_amount status branch_name
       6          1 2024-01-07    Customer_6    9800000006          FSD      45000.0          27696.0         17304.0   Open     Chennai
      11          1 2024-01-12   Customer_11    9800000011           DA      35000.0          24911.0         10089.0   Open     Chennai
      23          1 2024-01-24   Customer_23    9800000023           BA      30000.0          23616.0          6384.0   Open     Chennai
      35          1 2024-02-05   Customer_35    9800000035           BA      30000.0           7216.0         22784.0   Open     Chennai
      36          1 2024-02-06   Customer_36    9800000036           AI      48000.0          41359.0          6641.0   Open     Chennai
      57          1 2024-02-27   Customer_57    9800000057           ML      42000.0          36896.0          5104.0   Open     Che

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [69]:
# Query 17: Top 3 highest gross sales
query17 = """SELECT cs.*, b.branch_name
            FROM customer_sales cs
            JOIN branches b ON cs.branch_id = b.branch_id
            ORDER BY cs.gross_sales DESC
            LIMIT 3;"""
run_query(query17, "Query 17: Top 3 Highest Gross Sales")


📊 Query 17: Top 3 Highest Gross Sales
 sale_id  branch_id       date        name mobile_number product_name  gross_sales  received_amount  pending_amount status branch_name
      12          7 2024-01-13 Customer_12    9800000012           AI      48000.0          23283.0         24717.0   Open     Kolkata
      14          4 2024-01-15 Customer_14    9800000014           AI      48000.0          41943.0          6057.0   Open       Delhi
      17          6 2024-01-18 Customer_17    9800000017           AI      48000.0           9150.0         38850.0   Open        Pune

✅ Rows returned: 3


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [70]:
# Query 18: Branch with highest total gross sales
query18 = """SELECT b.branch_name, SUM(cs.gross_sales) AS total_gross_sales
            FROM branches b
            JOIN customer_sales cs ON b.branch_id = cs.branch_id
            GROUP BY b.branch_name
            ORDER BY total_gross_sales DESC
            LIMIT 1;"""
run_query(query18, "Query 18: Branch with Highest Gross Sales")


📊 Query 18: Branch with Highest Gross Sales
branch_name  total_gross_sales
    Kolkata          5101000.0

✅ Rows returned: 1


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [71]:
# Query 19: Monthly sales summary (group by month & year)
query19 = """SELECT YEAR(date) AS year, MONTH(date) AS month, 
                    COUNT(*) AS total_sales,
                    SUM(gross_sales) AS total_gross,
                    SUM(received_amount) AS total_received,
                    SUM(pending_amount) AS total_pending
            FROM customer_sales
            GROUP BY YEAR(date), MONTH(date)
            ORDER BY year DESC, month DESC;"""
run_query(query19, "Query 19: Monthly Sales Summary")


📊 Query 19: Monthly Sales Summary
 year  month  total_sales  total_gross  total_received  total_pending
 2024     12           60    2175000.0       1290457.0       884543.0
 2024     11           60    2248000.0       1087942.0      1160058.0
 2024     10           62    2362000.0       1247401.0      1114599.0
 2024      9           87    3183000.0       1629030.0      1553970.0
 2024      8           93    3254000.0       1639557.0      1614443.0
 2024      7           93    3517000.0       1681616.0      1835384.0
 2024      6           90    3294000.0       1783417.0      1510583.0
 2024      5           93    3418000.0       1745270.0      1672730.0
 2024      4           90    3272000.0       1901249.0      1370751.0
 2024      3           93    3442000.0       1798057.0      1643943.0
 2024      2           87    3116000.0       1489215.0      1626785.0
 2024      1           92    3487000.0       1734902.0      1752098.0

✅ Rows returned: 12


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [72]:
# Query 20: Payment method-wise total collection
query20 = """SELECT payment_method, SUM(amount_paid) AS total_collection
            FROM payment_splits
            GROUP BY payment_method
            ORDER BY total_collection DESC;"""
run_query(query20, "Query 20: Payment Method-wise Collection")


📊 Query 20: Payment Method-wise Collection
payment_method  total_collection
          Card         6608008.0
          Cash         6387006.0
           UPI         6033099.0

✅ Rows returned: 3


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [73]:
# CUSTOM QUERY 1: Sales by Product Category
# Shows total sales for each product
query_custom1 = """
SELECT 
    product_name,
    COUNT(*) as number_of_sales,
    SUM(gross_sales) as total_revenue
FROM customer_sales
GROUP BY product_name
ORDER BY total_revenue DESC;
"""

run_query(query_custom1, "Sales by Product Category")


📊 Sales by Product Category
product_name  number_of_sales  total_revenue
          AI              133      6384000.0
         FSD              127      5715000.0
          ML              127      5334000.0
          DA              133      4655000.0
          DS              114      4560000.0
          BI              140      3920000.0
          BA              110      3300000.0
         SQL              116      2900000.0

✅ Rows returned: 8


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [74]:
# CUSTOM QUERY 2: Branch Performance Summary
# Shows sales count and revenue per branch
query_custom2 = """
SELECT 
    b.branch_name,
    COUNT(cs.sale_id) as total_sales,
    SUM(cs.gross_sales) as total_revenue,
    SUM(cs.pending_amount) as total_pending
FROM branches b
LEFT JOIN customer_sales cs ON b.branch_id = cs.branch_id
GROUP BY b.branch_name
ORDER BY total_revenue DESC;
"""

run_query(query_custom2, "Branch Performance Summary")



📊 Branch Performance Summary
branch_name  total_sales  total_revenue  total_pending
    Kolkata          133      5101000.0      2639584.0
      Delhi          134      4914000.0      2580896.0
     Mumbai          136      4853000.0      2203172.0
  Bangalore          133      4710000.0      2315291.0
  Ahmedabad          128      4695000.0      2212011.0
       Pune          120      4558000.0      2156215.0
    Chennai          110      4128000.0      1852252.0
  Hyderabad          106      3809000.0      1780466.0

✅ Rows returned: 8


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [75]:
# CUSTOM QUERY 3: Open Sales with Pending Amount
# Shows all sales that are still open
print("\n" + "="*80)
print("🟡 CUSTOM QUERY 3: Open Sales with Pending Amount")
print("="*80)

query_custom3 = """
SELECT 
    sale_id,
    name,
    gross_sales,
    received_amount,
    pending_amount,
    status
FROM customer_sales
WHERE status = 'Open'
ORDER BY pending_amount DESC;
"""

run_query(query_custom3, "Open Sales (Pending Payments)")


🟡 CUSTOM QUERY 3: Open Sales with Pending Amount

📊 Open Sales (Pending Payments)
 sale_id          name  gross_sales  received_amount  pending_amount status
     598  Customer_598      48000.0             44.0         47956.0   Open
     369  Customer_369      48000.0            339.0         47661.0   Open
     748  Customer_748      48000.0            369.0         47631.0   Open
     658  Customer_658      48000.0            554.0         47446.0   Open
     244  Customer_244      48000.0           1228.0         46772.0   Open
     650  Customer_650      48000.0           2002.0         45998.0   Open
     648  Customer_648      48000.0           2167.0         45833.0   Open
     780  Customer_780      45000.0            316.0         44684.0   Open
     426  Customer_426      45000.0            418.0         44582.0   Open
     859  Customer_859      45000.0            638.0         44362.0   Open
     121  Customer_121      45000.0            665.0         44335.0   Open
     

C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [76]:
# CUSTOM QUERY 4: Payment Method Summary
# Shows total collection by payment type
print("\n" + "="*80)
print("💳 CUSTOM QUERY 4: Payment Method Summary")
print("="*80)

query_custom4 = """
SELECT 
    payment_method,
    COUNT(*) as number_of_payments,
    SUM(amount_paid) as total_amount
FROM payment_splits
GROUP BY payment_method
ORDER BY total_amount DESC;
"""

run_query(query_custom4, "Payment Method Analysis")


💳 CUSTOM QUERY 4: Payment Method Summary

📊 Payment Method Analysis
payment_method  number_of_payments  total_amount
          Card                 695     6608008.0
          Cash                 669     6387006.0
           UPI                 647     6033099.0

✅ Rows returned: 3


C:\Users\rvdhi\AppData\Local\Temp\ipykernel_20388\1267830458.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, conn)


In [ ]:
print("\n" + "="*80)
print("📈 BUSINESS INSIGHTS SUMMARY")
print("="*80)


📈 BUSINESS INSIGHTS SUMMARY


In [20]:
print("\n" + "="*80)
print("📈 BUSINESS INSIGHTS SUMMARY")
print("="*80)

conn = get_connection()
cursor = conn.cursor()

cursor.execute("SELECT SUM(gross_sales), SUM(received_amount), SUM(pending_amount) FROM customer_sales")
total_gross, total_received, total_pending = cursor.fetchone()

print(f"\n💰 Total Gross Sales: ₹{total_gross:,.2f}")
print(f"✅ Total Received: ₹{total_received:,.2f}")
print(f"⏳ Total Pending: ₹{total_pending:,.2f}")
print(f"📊 Collection Rate: {(total_received/total_gross)*100:.2f}%")

print("\n🏆 Branch Performance Ranking:")
cursor.execute("""
    SELECT b.branch_name, COUNT(cs.sale_id) as sales, SUM(cs.gross_sales) as revenue
    FROM branches b LEFT JOIN customer_sales cs ON b.branch_id = cs.branch_id
    GROUP BY b.branch_name ORDER BY revenue DESC;
""")
for i, row in enumerate(cursor.fetchall(), 1):
    revenue = row[2] if row[2] else 0
    print(f"   {i}. {row[0]}: ₹{revenue:,.2f} | Sales: {row[1]}")

print("\n💳 Payment Method Analysis:")
cursor.execute("""
    SELECT payment_method, COUNT(*) as transactions, SUM(amount_paid) as total
    FROM payment_splits GROUP BY payment_method ORDER BY total DESC;
""")
for row in cursor.fetchall():
    print(f"   • {row[0]}: {row[1]} transactions | ₹{row[2]:,.2f}")

cursor.close()
conn.close()

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE!")
print("="*80)


📈 BUSINESS INSIGHTS SUMMARY

💰 Total Gross Sales: ₹36,768,000.00
✅ Total Received: ₹19,028,113.00
⏳ Total Pending: ₹17,739,887.00
📊 Collection Rate: 51.75%

🏆 Branch Performance Ranking:
   1. Kolkata: ₹5,101,000.00 | Sales: 133
   2. Delhi: ₹4,914,000.00 | Sales: 134
   3. Mumbai: ₹4,853,000.00 | Sales: 136
   4. Bangalore: ₹4,710,000.00 | Sales: 133
   5. Ahmedabad: ₹4,695,000.00 | Sales: 128
   6. Pune: ₹4,558,000.00 | Sales: 120
   7. Chennai: ₹4,128,000.00 | Sales: 110
   8. Hyderabad: ₹3,809,000.00 | Sales: 106

💳 Payment Method Analysis:
   • Card: 695 transactions | ₹6,608,008.00
   • Cash: 669 transactions | ₹6,387,006.00
   • UPI: 647 transactions | ₹6,033,099.00

✅ ANALYSIS COMPLETE!


In [22]:
print("\n" + "="*80)
print("🔐 LOGIN CREDENTIALS")
print("="*80)


🔐 LOGIN CREDENTIALS


In [ ]:
print("\n" + "="*60)
print("🔐 LOGIN CREDENTIALS")
print("="*60)

conn = get_connection()
df_users = pd.read_sql("SELECT username, role, email FROM users", conn)
print(df_users.to_string(index=False))
conn.close()

print("\n📝 Passwords:")
print("   Super Admin: super123")
print("   All Branch Admins: admin123")

print("\n" + "="*60)
print("✅ NOTEBOOK COMPLETE!")
print("="*60)


🔐 LOGIN CREDENTIALS
       username        role                  email
     superadmin Super Admin superadmin@company.com
  admin_chennai       Admin    chennai@company.com
admin_bangalore       Admin  bangalore@company.com
admin_hyderabad       Admin  hyderabad@company.com
    admin_delhi       Admin      delhi@company.com
   admin_mumbai       Admin     mumbai@company.com
     admin_pune       Admin       pune@company.com
  admin_kolkata       Admin    kolkata@company.com
admin_ahmedabad       Admin  ahmedabad@company.com

📝 Passwords:
   Super Admin: super123
   All Branch Admins: admin123

✅ NOTEBOOK COMPLETE!
